# Train the "important points" model

This notebook trains the small AI model that picks out the important sentences in
the Video Compressor's **Transcribe & notes** tool. When it's done you get a folder
of about 14 MB that the web page runs on your own device, with nothing uploaded.

**Before you start:** choose **Runtime → Change runtime type → T4 GPU**, then
**Runtime → Run all**. The whole notebook takes about 30–60 minutes on the free GPU.

What it does:
1. Downloads public datasets of long texts with human-written summaries
   (MeetingBank meeting transcripts and CNN/DailyMail news).
2. Labels each sentence as important if it best matches the summary.
3. Fine-tunes ELECTRA-small to recognise those sentences.
4. Compares it with the page's current rules on documents it never saw.
5. Exports it in the format the page loads, and downloads it as a zip.

In [ ]:
# Get the training code and install what Colab doesn't already have.
!git clone -q --depth 1 https://github.com/rhhas-knust/Video-Compressor.git
%cd Video-Compressor/training
!pip install -q -r requirements.txt

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - choose Runtime > Change runtime type > T4 GPU")

## 1. Download and label the data
Lower `MEETINGS` and `NEWS` for a quicker first try (for example 200 and 1000);
higher values train a better model but take longer.

In [ ]:
import notes_model as nm

MEETINGS = 1200   # MeetingBank transcripts (closest to lectures and talks)
NEWS = 6000       # CNN/DailyMail articles

docs = nm.load_documents(max_meetings=MEETINGS, max_news=NEWS)
rows = nm.make_examples(docs["train"])
print(f"{len(rows)} training sentences, {sum(r['label'] for r in rows)} of them important")
print("Example:", rows[0])

## 2. Train

In [ ]:
nm.train(rows, "notes-model-pt", epochs=2)

## 3. Is it better than the current rules?
ROUGE measures how much of the human summary the picked sentences cover (higher is
better). `trained model` should beat `current rules` before you use it on the site.

In [ ]:
import pandas as pd

table = nm.evaluate(docs["test"], {
    "trained model": nm.model_scorer("notes-model-pt"),
    "current rules": nm.rule_scores,
    "first sentences": lambda s: [1.0 / (i + 1) for i in range(len(s))],
})
pd.DataFrame(table)

In [ ]:
# Try it on your own text: paste a transcript between the quotes.
text = """Today we are going to look at how vaccines work. The most important thing to remember is that a vaccine trains the immune system without causing the disease. Um, let me find my slides. A vaccine contains a weakened or inactive part of a germ, called an antigen. The immune system makes antibodies against the antigen. If the real germ arrives later, those antibodies recognise it quickly. That's why vaccinated people usually get much less sick. Okay, any questions so far?"""

sentences = nm.split_sentences(text)
scores = nm.model_scorer("notes-model-pt")(sentences)
for p, s in sorted(zip(scores, sentences), reverse=True):
    print(f"{p:.2f}  {s}")

## 4. Export for the web page and download

In [ ]:
nm.export_onnx("notes-model-pt", "notes-model")
!cd /content/Video-Compressor/training && zip -qr notes-model.zip notes-model && ls -la notes-model.zip

from google.colab import files
files.download("notes-model.zip")

## 5. Add it to the site
1. Unzip `notes-model.zip`. You get a `notes-model` folder.
2. On GitHub, open the repository, go to `web/models/`, click **Add file → Upload files**,
   drag the whole `notes-model` folder in, and commit.
3. About a minute later the site uses it. After a transcription, the notes say
   *"Picked by the trained notes model on your device."*

The datasets are for research and personal use; check their licences before using
the model commercially.